In [5]:
# %% 
import time, os, json
from pathlib import Path
import pandas as pd
import numpy as np
import itertools
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df
)

# %% 1) Basis-Pfade & Hyperparameter-Räume
splits_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits")
models_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/models")
output_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data")
output_root.mkdir(parents=True, exist_ok=True)

feature_counts    = [3,10, 15,50]#, 20, 50, 100, 200]
selection_methods = ["random_forest","forward","univariate","variance","lasso"]
reducers          = ["pca","lda","umap"]
n_components_list = [2, 3, 5, 10]
outlier_methods   = [None,"clip","zscore"]
scaler_methods    = [None,"standard","robust"]
use_sex_opts      = [False, True]
n_jobs            = -1

# %% 2) Grid erzeugen & 100 auswählen
grid = pd.DataFrame(
    itertools.product(
        feature_counts,
        selection_methods,
        reducers,
        n_components_list,
        outlier_methods,
        scaler_methods,
        use_sex_opts
    ),
    columns=[
        "feature_count",
        "selection_method",
        "reducer",
        "n_components",
        "outlier_method",
        "scaler_method",
        "use_sex"
    ]
).sort_values("feature_count", ascending=False).reset_index(drop=True)

sampled_grid = grid.sample(n=10, random_state=42)
sampled_grid = sampled_grid.sort_values("feature_count", ascending=False).reset_index(drop=True)

# %% 3) Spezies-Schleife
for species_dir in sorted(splits_root.iterdir()):
    if not species_dir.is_dir(): 
        continue
    species = species_dir.name
    print(f"\n🔄 Processing species: {species}")

    # 3.1) Daten laden
    df = pd.read_csv(species_dir / "train.csv")

    # 3.2) Feature-Spalten wählen
    if species == "eurasian_otter_(1)":
        feature_cols = [
            c for c in df.columns 
            if c.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[c])
        ]
    else:
        feature_cols = [
            c for c in df.columns[5:]
            if pd.api.types.is_numeric_dtype(df[c])
        ]
    print(f"  → {len(feature_cols)} numerische Features")

    # 3.3) Pairs & Summary generieren
    comps, summary = generate_pairwise_comparisons_from_df(
        df,
        id_col="individual_id",
        chunk_size=10,
        trail_size_list=[10,7,5,3],
        max_individuals=5,
        max_trails_per_animal=2,
        n_folds=5,
        random_state=42
    )
    print("  Summary der Trail-Längen:")
    display(summary)

    # 3.4) Resume-Logik für species-spezifisches Parquet
    results_fp = output_root / f"{species}_all_results_combined.parquet"
    if results_fp.exists():
        done = pd.read_parquet(results_fp, columns=sampled_grid.columns)
        done_keys = set(tuple(r) for r in done.values)
        todo = sampled_grid[
            sampled_grid.apply(lambda row: tuple(row.values) not in done_keys, axis=1)
        ].reset_index(drop=True)
    else:
        todo = sampled_grid.copy()

    print(f"  → Noch offen: {len(todo)}/{len(sampled_grid)} Runs")

    # 3.5) Hyper-Parameter-Loop
    for idx, row in todo.iterrows():
        fc, sel, red, nc, out, scl, sex = (
            int(row.feature_count),
            row.selection_method,
            row.reducer,
            int(row.n_components),
            row.outlier_method,
            row.scaler_method,
            bool(row.use_sex)
        )
        desc = (
            f"FC={fc}|FS={sel}|Red={red}/{nc}|Out={out}|Scl={scl}|Sex={'on' if sex else 'off'}"
        )
        print(f"    → Run {idx+1}/{len(todo)}: {desc}")
        t0 = time.time()

        with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
            res = run_all_pairwise_projections_parallel(
                comparisons=comps,
                df=df,
                feature_cols=feature_cols,
                k_features=fc,
                selection_method=sel,
                reducers=[red],
                n_components=[nc],
                outlier_methods=out,
                scaler_methods=scl,
                use_sexmodel_prediction=sex,
                sexmodel_path=(
                    str(models_root / f"{species}.joblib")
                    if sex else None
                ),
                debug=False,
                n_jobs=n_jobs
            )
        dur = time.time() - t0
        print(f"      → {len(res)} Rows in {dur:.1f}s")

        # 3.6) Results anreichern & speichern
        df_run = pd.DataFrame(res)
        for col in sampled_grid.columns:
            df_run[col] = row[col]
        if results_fp.exists():
            df_full = pd.concat([pd.read_parquet(results_fp), df_run], ignore_index=True)
        else:
            df_full = df_run
        df_full.to_parquet(results_fp, index=False)

    print(f"  ✅ Fertig für {species}, Parquet: {results_fp}")

# %% 4) Alle Spezies vereinen
all_parquets = list(output_root.glob("*_all_results_combined.parquet"))
df_all = pd.concat((pd.read_parquet(p) for p in all_parquets), ignore_index=True)
df_all.to_parquet(output_root / "all_species_all_results_combined.parquet", index=False)
print("\n🎉 Fertig – Gesamt-Parquet erzeugt:")
print(f"   {output_root/'all_species_all_results_combined.parquet'}")
display(df_all.head())



🔄 Processing species: aj_someringii
  → 137 numerische Features
  Summary der Trail-Längen:


,sub_size,n_animals,n_trails,avg_comp_per_ind_same,sd_comp_per_ind_same,avg_comp_per_ind_diff,sd_comp_per_ind_diff,ratio_same_to_diff
0,10,4,5,NaN,NaN,NaN,NaN,NaN
1,7,5,6,NaN,NaN,NaN,NaN,NaN
2,5,5,6,NaN,NaN,NaN,NaN,NaN
3,3,5,9,NaN,NaN,NaN,NaN,NaN
4,Total,5,97,5.6,6.0663,27.6,6.387488,0.405797


  → Noch offen: 10/10 Runs
    → Run 1/10: FC=50|FS=univariate|Red=umap/2|Out=zscore|Scl=None|Sex=on


FC=50|FS=univariate|Red=umap/2|Out=zscore|Scl=None|Sex=on:   0%|          | 0/97 [00:00<?, ?it/s]

  0%|          | 0/97 [00:00<?, ?it/s]

...

Processing Pairs:   0%|          | 0/192 [00:22<?, ?it/s]




















  0%|          | 0/192 [00:00<?, ?it/s]

      → 192 Rows in 0.7s
  ✅ Fertig für white_rhino, Parquet: C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\white_rhino_all_results_combined.parquet

🎉 Fertig – Gesamt-Parquet erzeugt:
   C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\all_species_all_results_combined.parquet


Processing Pairs:   0%|          | 0/192 [00:01<?, ?it/s]


,trail_a_id,trail_b_id,samples_a,samples_b,ind_a,ind_b,same_individual,fold,pipeline,selection_method,...,coords_r_y,dist_euclidean,dist_manhattan,dist_cosine,dist_chebyshev,dist_canberra,dist_braycurtis,dist_mahalanobis,feature_count,use_sex
0,adaeze_7c0,1986_7c0,"[198, 196, 195, 194, 192, 199, 197]","[140, 122, 138, 132, 126, 127, 131]",adaeze,1986,False,0,univariate_zscore_no_scale_k50_umap_nc2_sex_on,univariate,...,"[-8.144471168518066, -10.304457664489746, -10....",7.860942,8.763704,1.100513,7.801878,1.139728,1.010141,NaN,50,True
1,adaeze_7c0,1986_7c1,"[198, 196, 195, 194, 192, 199, 197]","[123, 137, 125, 297, 136, 134, 135]",adaeze,1986,False,2,univariate_zscore_no_scale_k50_umap_nc2_sex_on,univariate,...,"[13.3018217086792, 13.275851249694824, 13.0458...",4.185018,5.858797,0.013199,3.348694,0.430499,0.167374,NaN,50,True
2,adaeze_5c0,1986_5c0,"[194, 192, 199, 198, 193]","[132, 140, 126, 124, 127]",adaeze,1986,False,4,univariate_zscore_no_scale_k50_umap_nc2_sex_on,univariate,...,"[0.4370776116847992, 3.715266466140747, 1.1549...",3.248589,4.029478,0.158000,3.118140,0.808847,0.424583,NaN,50,True
3,adaeze_5c0,1986_5c1,"[194, 192, 199, 198, 193]","[123, 139, 137, 297, 125]",adaeze,1986,False,4,univariate_zscore_no_scale_k50_umap_nc2_sex_on,univariate,...,"[4.575276851654053, 4.58577299118042, 4.523488...",4.158275,5.854321,0.023479,3.205293,1.004056,0.470880,NaN,50,True
4,adaeze_3c0,1986_3c0,"[198, 193, 197]","[122, 133, 129]",adaeze,1986,False,2,univariate_zscore_no_scale_k50_umap_nc2_sex_on,univariate,...,"[1.7618720531463623, 2.093327522277832, 0.1372...",4.395805,5.411736,0.474893,4.235518,1.542002,0.841440,NaN,50,True


In [ ]:
# %% 
import time
import os
import pandas as pd
import random
import json
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel,
    generate_pairwise_comparisons_from_df
)

# --- Basis-Pfade ---
splits_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits")
models_root = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/models")
output_root = Path("D:/Projects and Disschaptors/FIT_package/results/data")
output_root.mkdir(parents=True, exist_ok=True)

# --- Gemeinsame Parameter ---
feature_counts     = [10,12,16, 15,16, 20, 50, 100]
selection_methods  = ["random_forest","forward"]
reducers           = ["pca", "lda", "umap"]
n_components_list  = [2, 3, 4]
n_jobs             = -1

# Zum globalen Zusammenbau
all_species_results = []

# %%  
# Schleife über alle Species-Ordner
for species_dir in sorted(splits_root.iterdir()):
    if not species_dir.is_dir():
        continue
    species_name = species_dir.name
    print(f"\n🔄 Processing species: {species_name}")

    # 1) Daten einlesen
    df = pd.read_csv(species_dir / "train.csv")

    # 2) Feature-Spalten auswählen
    # Für Eurasian Otter bleiben wir bei dist/ang/t…-Columns
    if species_name == "eurasian_otter_(1)":
        feature_cols = [
            col for col in df.columns
            if col.startswith(("dist","ang","t")) and pd.api.types.is_numeric_dtype(df[col])
        ]
    else:
        # alle numerischen ab 6. Spalte
        feature_cols = [
            col for col in df.columns[5:]
            if pd.api.types.is_numeric_dtype(df[col])
        ]
    print(f"  → {len(feature_cols)} numerische Features gefunden.")

    # 3) Paarvergleiche generieren (gleiche Einstellungen für alle Species)
    comparisons = generate_pairwise_comparisons_from_df(
        df,
        id_col="individual_id",
        group_sizes=[2,3,5,7],
        n_repeats=3,
        mode="both",
        selfmatch_factor=10
    )
    print(f"  → {len(comparisons)} raw pairs generated.")
   # comparisons =random.sample(comparisons, 20)
    
    # 4) Sweep nur über selection_methods
    species_results = []
    timing_log = []

    for sel in tqdm(selection_methods, desc=f"FS methods for {species_name}"):
        t0_sel = time.time()
        for use_sex in (False, True):
            tag = "sex_on" if use_sex else "sex_off"
            # Pfad zum Sex-Modell
            sexmodel_path = models_root / f"{species_name}.joblib" if use_sex else None

            with tqdm_joblib(tqdm(desc=f"Pairs ({sel} | {tag})", total=len(comparisons))):
                results = run_all_pairwise_projections_parallel(
                    comparisons=comparisons,
                    df=df,
                    feature_cols=feature_cols,
                    k_features=feature_counts,
                    reducers=reducers,
                    selection_method=sel,
                    n_components=n_components_list,
                    use_sexmodel_prediction=use_sex,
                    sexmodel_path=str(sexmodel_path) if sexmodel_path else None,
                    debug=False,
                    n_jobs=n_jobs
                )
            dur = time.time() - t0_sel
            print(f"    • {sel} | sex={use_sex}: {len(results)} rows in {dur:.1f}s")

            # DataFrame speichern
            df_res = pd.DataFrame(results)
            df_res["selection_method"]        = sel
            df_res["use_sexmodel_prediction"] = use_sex
            df_res["species"]                 = species_name
            fname = output_root / f"{species_name}_pairwise_{sel}_{tag}_{len(comparisons)}pairs.csv"
            df_res.to_csv(fname, index=False)

            species_results.append(df_res)
            timing_log.append({
                "species":                species_name,
                "selection_method":       sel,
                "use_sexmodel_prediction":use_sex,
                "duration_s":             dur,
                "n_results":              len(results)
            })

    # pro Species auch Gesamt-CSV
    if species_results:
        sp_full = pd.concat(species_results, ignore_index=True)
        sp_full.to_csv(output_root / f"{species_name}_all_results.csv", index=False)
        print(f"  ✔️ Wrote species-combined results for {species_name}")

    # Timing speichern
    pd.DataFrame(timing_log).to_csv(output_root / f"{species_name}_timing_log.csv", index=False)

    all_species_results.append(sp_full)

# %%  
# --- Globales Zusammenführen über alle Species ---
if all_species_results:
    all_full = pd.concat(all_species_results, ignore_index=True)
    all_full.to_csv(output_root / "all_species_all_results_combined.csv", index=False)
    print("\n🎉 Alle Species kombiniert gespeichert in 'all_species_all_results_combined.csv'")
